In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

metadata_df

,sample_title,disease state,tissue,treatment
samples,,,,
GSM6077322,sample100,hepatocellular carcinoma,tumor,without treatment
GSM6077323,sample102,hepatocellular carcinoma,tumor,without treatment
GSM6077324,sample104,hepatocellular carcinoma,tumor,without treatment
GSM6077325,sample106,hepatocellular carcinoma,tumor,without treatment
GSM6077326,sample109,hepatocellular carcinoma,tumor,without treatment
GSM6077327,sample110,hepatocellular carcinoma,tumor,without treatment
GSM6077328,sample114,hepatocellular carcinoma,tumor,without treatment
GSM6077329,sample115,hepatocellular carcinoma,tumor,without treatment
GSM6077330,sample119,hepatocellular carcinoma,tumor,without treatment


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_4468/4076848676.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,disease state,tissue,treatment,DDX11L1,WASH7P,MIR6859-1,MIR1302-2HG,MIR1302-2,FAM138A,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM6077322,sample100,hepatocellular carcinoma,tumor,without treatment,3,596,27,2,0,0,...,198783,18,15,21,31864,13596,1715,56720,102,649
GSM6077323,sample102,hepatocellular carcinoma,tumor,without treatment,1,494,9,0,0,0,...,187557,13,11,15,37022,22063,2871,67004,61,836
GSM6077324,sample104,hepatocellular carcinoma,tumor,without treatment,1,516,11,1,2,0,...,165290,16,10,11,36979,17879,1605,76678,136,593
GSM6077325,sample106,hepatocellular carcinoma,tumor,without treatment,5,646,18,0,0,4,...,199041,13,27,30,12486,6283,417,56701,45,97
GSM6077326,sample109,hepatocellular carcinoma,tumor,without treatment,7,369,7,1,1,0,...,376862,31,25,26,49755,17909,2291,139270,72,1505
GSM6077327,sample110,hepatocellular carcinoma,tumor,without treatment,1,461,4,3,1,0,...,167035,13,28,33,30916,33012,3512,93740,170,902
GSM6077328,sample114,hepatocellular carcinoma,tumor,without treatment,1,700,20,0,1,0,...,156897,15,25,28,39919,25280,2422,81857,90,1234
GSM6077329,sample115,hepatocellular carcinoma,tumor,without treatment,1,1090,20,1,0,3,...,166110,13,30,30,19761,11926,809,62016,69,936
GSM6077330,sample119,hepatocellular carcinoma,tumor,without treatment,0,826,43,0,0,1,...,221308,97,161,153,31746,10625,892,71765,85,359


In [3]:
import pandas as pd

# Download supplementary data from: https://pmc.ncbi.nlm.nih.gov/articles/instance/9428591/bin/Table_1.xlsx

# Read specific sheet (sheet number 9, which is index 8)
df_clinical_data = pd.read_excel('Table_1.xlsx', sheet_name=8, skiprows=1)
df_clinical_data.to_csv('Table_1_clinical_data.csv', index=False)


# Merge the dataframes
df_combined = df_clinical_data.merge(
    df_combined,
    left_on='Sample.ID',
    right_on='sample_title',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
# drop the Sample.ID column
df_combined = df_combined.drop(columns=['Sample.ID'])
df_combined.to_csv(data_file_name, index=False)

df_combined

,Hypoxia_DEG_Score,OS,OS.time,Gender,Grouping,sample_title,disease state,tissue,treatment,DDX11L1,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,7.195420,1,2.504110,Male,High,sample49,hepatocellular carcinoma,tumor,without treatment,3,...,241190,33,52,70,41553,15159,1171,84715,67,993
1,46.831832,1,0.706849,Male,High,sample67,hepatocellular carcinoma,tumor,without treatment,1,...,130819,65,90,92,29113,14814,1330,47474,136,865
2,47.358300,0,5.087671,Male,High,sample68,hepatocellular carcinoma,tumor,without treatment,4,...,101367,103,147,142,11896,4366,146,35574,112,535
3,-26.489002,0,4.980822,Male,Low,sample69,hepatocellular carcinoma,tumor,without treatment,1,...,343984,32,27,35,118537,69954,5273,136930,206,1817
4,-45.116475,0,4.975342,Male,Low,sample70,hepatocellular carcinoma,tumor,without treatment,1,...,201978,164,154,180,74744,35165,2691,158878,93,756
5,-16.883033,1,3.210959,Male,High,sample72,hepatocellular carcinoma,tumor,without treatment,8,...,247035,171,223,224,50894,32011,3127,136980,144,1407
6,-19.032081,0,4.964384,Male,Low,sample73,hepatocellular carcinoma,tumor,without treatment,2,...,175030,108,153,159,27424,15778,1400,78133,109,786
7,-2.147005,0,0.106849,Male,High,sample74,hepatocellular carcinoma,tumor,without treatment,0,...,233232,83,79,84,95867,42506,3874,140385,95,1221
8,16.126034,0,4.964384,Male,High,sample76,hepatocellular carcinoma,tumor,without treatment,5,...,108632,169,201,522,29976,12789,1610,37118,107,513
9,-9.220302,0,0.112329,Male,High,sample78,hepatocellular carcinoma,tumor,without treatment,4,...,96254,57,74,77,30570,17276,2125,48336,50,412


# Generate description


In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'Table_1_clinical_data.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_clinical_data.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
